# Dual-heuristic post-processing pipeline

This notebook runs the thesis dual-heuristic pipeline on an existing ByteTrack text file. It keeps the tracker output frozen, extracts representative face crops, embeds each tracklet with the original ChimpUFE backbone, chooses the number of anonymous identities with the physical lower bound plus silhouette search, and writes a clustered `#`-separated prediction file.

The notebook is intentionally written as a compact runbook: configure paths, run the pipeline, then inspect the selected `K`, silhouette scores, and distance matrix.

## Configuration

Edit the paths below to point to your video and model files. See [SETUP.md](SETUP.md) for model weight download instructions.

In [1]:
import json
import sys
from pathlib import Path

try:
    NOTEBOOK_DIR = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path.cwd().resolve()

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from dual_heuristic_pipeline import DualHeuristicConfig, run_dual_heuristic

PROJECT_ROOT = NOTEBOOK_DIR.parent.parent
RESULTS_DIR = NOTEBOOK_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"notebook dir : {NOTEBOOK_DIR}")
print(f"project root : {PROJECT_ROOT}")

notebook dir : /home/diego/Desktop/ChimpRec/ChimpRec/Code/PostProcessingDualHeuristic
project root : /home/diego/Desktop/ChimpRec/ChimpRec


In [ ]:
# ===== CONFIGURATION =====
# Edit these paths to match your environment

# Video and track file
VIDEO_NAME = "20241104 - 11h33"  # Change to your video name
VIDEO_PATH = PROJECT_ROOT / "ChimpVideos" / "input" / f"{VIDEO_NAME}.MP4"
TRACKS_TXT = PROJECT_ROOT / "ChimpVideos" / "output" / "temp" / "raw_output" / f"{VIDEO_NAME}.txt"

# Model weights directory (download from Google Drive link in SETUP.md)
WEIGHTS_DIR = NOTEBOOK_DIR / "weights"
CHIMPUFE_ROOT = NOTEBOOK_DIR / "ChimpUFE"
CHIMPUFE_WEIGHTS = WEIGHTS_DIR / "25-08-29T11-49-28_340k.pth"
FACE_MODEL = WEIGHTS_DIR / "yolox_best_only_model.pth"

# Output directory
OUTPUT_TXT = RESULTS_DIR / f"{VIDEO_NAME}_dualheuristic.txt"
DIAGNOSTICS_JSON = RESULTS_DIR / f"{VIDEO_NAME}_dualheuristic.json"
DISTANCE_MATRIX_NPY = RESULTS_DIR / f"{VIDEO_NAME}_distance_matrix.npy"
CROPS_DIR = RESULTS_DIR / f"{VIDEO_NAME}_selected_crops"

# Pipeline configuration
config = DualHeuristicConfig(
    samples_per_track=5,
    crop_pool_size=30,
    min_track_len=30,
    max_cluster_k=20,
    cluster_prefix="cluster_",
    unassigned_policy="unique",
    use_spatial_recovery=False,
    device="auto",
)

# Verify all paths exist
for label, path in {
    "video": VIDEO_PATH,
    "tracks": TRACKS_TXT,
    "ChimpUFE root": CHIMPUFE_ROOT,
    "ChimpUFE weights": CHIMPUFE_WEIGHTS,
    "face model": FACE_MODEL,
}.items():
    print(f"{label:16s}: {path}  exists={path.exists()}")

video           : /home/diego/Desktop/ChimpRec/ChimpRec/ChimpVideos/input/20241104 - 11h33.MP4  exists=True
tracks          : /home/diego/Desktop/ChimpRec/ChimpRec/ChimpVideos/output/temp/raw_output/20241104 - 11h33.txt  exists=True
ChimpUFE root   : /home/diego/Desktop/ChimpRec/ChimpRec/Code/PostProcessingDualHeuristic/ChimpUFE  exists=True
ChimpUFE weights: /home/diego/Desktop/ChimpRec/ChimpRec/Code/PostProcessingDualHeuristic/weights/25-08-29T11-49-28_340k.pth  exists=True
face model      : /home/diego/Desktop/ChimpRec/ChimpRec/Code/PostProcessingDualHeuristic/weights/yolox_best_only_model.pth  exists=True


## Run the post-processing

This cell performs crop extraction and model inference. It can take time on long videos, so run it only after the paths above are correct.

In [4]:
result = run_dual_heuristic(
    tracker_txt=TRACKS_TXT,
    output_txt=OUTPUT_TXT,
    video_path=VIDEO_PATH,
    face_model_path=FACE_MODEL,
    chimpufe_weights_path=CHIMPUFE_WEIGHTS,
    chimpufe_root=CHIMPUFE_ROOT,
    config=config,
    diagnostics_json=DIAGNOSTICS_JSON,
    distance_matrix_npy=DISTANCE_MATRIX_NPY,
    save_crops_dir=CROPS_DIR,
)

result.to_json_dict()

Dual-heuristic crops:   3%|▎         | 5/194 [14:18<9:00:43, 171.66s/it]


KeyboardInterrupt: 

## Optional: run from precomputed signatures

Use this route when you only want to debug clustering logic. The `.npz` file must contain `track_ids` and `signatures`.

In [ ]:
# SIGNATURES_NPZ = RESULTS_DIR / "precomputed_signatures.npz"
# result = run_dual_heuristic(
#     tracker_txt=TRACKS_TXT,
#     output_txt=RESULTS_DIR / f"{VIDEO_NAME}_dualheuristic_from_npz.txt",
#     signatures_npz=SIGNATURES_NPZ,
#     config=config,
#     diagnostics_json=RESULTS_DIR / f"{VIDEO_NAME}_dualheuristic_from_npz.json",
#     distance_matrix_npy=RESULTS_DIR / f"{VIDEO_NAME}_distance_matrix_from_npz.npy",
# )
# result.to_json_dict()

## Diagnostics

Inspect the selected cluster count and visualize the pairwise distance matrix.

In [ ]:
with open(DIAGNOSTICS_JSON) as handle:
    diagnostics = json.load(handle)

print(f"embedded tracklets   : {diagnostics['n_embedded_tracklets']}")
print(f"unembedded tracklets : {diagnostics['n_unembedded_tracklets']}")
print(f"K_min               : {diagnostics['k_min']}")
print(f"selected K          : {diagnostics['selected_k']}")
diagnostics['silhouette_scores']

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

distance_matrix = np.load(DISTANCE_MATRIX_NPY)
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(distance_matrix, cmap="viridis")
ax.set_title("Dual-heuristic cosine-distance matrix")
ax.set_xlabel("tracklet index")
ax.set_ylabel("tracklet index")
fig.colorbar(im, ax=ax, label="distance")
fig.tight_layout()